<a href="https://colab.research.google.com/github/antoine955/-Project-Submission-notebook/blob/main/Guided_Hands_on_Introduction_to_Model_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

## Business Context

In the competitive landscape of retail banking, customer retention is critical for ensuring sustainable growth and profitability. A prominent retail banking institution in Europe provides a range of financial products, including credit cards, loans, and savings accounts, and has been rapidly expanding its customer base across multiple countries. However, with a growing customer base, it faces an increasingly pressing challenge: customer churn. A significant number of customers are closing their accounts and switching to competitors. This decline in customer retention is impacting revenue and long-term customer relationships

Understanding the reasons behind customer attrition (or churn) is essential for the bank to devise effective retention strategies to minimize churn and enhance customer loyalty and satisfaction. The Customer Analytics & Retention Department has been diligently collecting and analyzing historical customer data. Despite the valuable insights provided by historical data, the department grapples with several challenges:

1. **Complex Customer Behavior**: The diverse nature of the bank's offerings and the varying customer preferences across different countries complicate the identification of factors that lead to churn.
2. **Proactive Retention**: The current processes for identifying at-risk customers are reactive rather than proactive, leading to missed opportunities for timely interventions that could prevent churn.

## Objective

The Retention Intelligence Unit has successfully conducted an in-depth analysis of customer data to uncover patterns influencing churn. Their findings highlight key factors contributing to customer attrition, providing an opportunity to implement targeted retention strategies.  

However, to effectively act on these insights, the team aims to operationalize them through a real-time decision-support system. The primary objective is to develop an interactive analytics platform that equips customer service teams, marketing departments, and business leaders with predictive churn insights. The successful deployment of this solution will enable timely interventions, enhance customer engagement efforts, and drive long-term customer retention.

## Data Dictionary

- **customerID**: Unique identifier for each customer.  
- **SeniorCitizen**: Indicates whether the customer is a senior citizen (binary: 0 = No, 1 = Yes).  
- **Partner**: Indicates if the customer has a partner (Yes/No).  
- **Dependents**: Indicates if the customer has dependents (Yes/No).  
- **tenure**: Number of months the customer has been with the service.  
- **PhoneService**: Indicates if the customer has a phone service (Yes/No).  
- **InternetService**: Type of internet service (DSL, Fiber optic, No).  
- **Contract**: Type of contract (Month-to-month, One year, Two year).  
- **PaymentMethod**: Method used for payment (Credit card, Electronic check, etc.).  
- **MonthlyCharges**: The monthly amount charged to the customer.  
- **TotalCharges**: The total amount charged to the customer over time.  
- **Churn**: Indicates whether the customer has left the service (Yes/No).  

# Installing and Importing Necessary Libraries

In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 xgboost==2.1.4 joblib==1.4.2 streamlit==1.43.2 huggingface_hub==0.29.3 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 50.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
# for data manipulation
import pandas as pd

import sklearn

# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score

# for model serialization
import joblib

# for creating a folder
import os

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

In [ ]:
# Set scikit-learn's display mode to 'diagram' for better visualization of pipelines and estimators
sklearn.set_config(display='diagram')

- `set_config` is a function in sklearn that changes global settings for visualization.
- `display='diagram'` enables a more detailed and structured graphical representation of pipelines when displaying the output in code cells.
- This is useful when working with `Pipeline` or `ColumnTransformer` objects to visualize preprocessing steps and model flow clearly.

# Data Loading and Overview

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Code to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load the dataset from a CSV file into a Pandas DataFrame
telecom_churn = pd.read_csv("/content/drive/MyDrive/customer_churn_data.csv")

In [ ]:
# Create a copy of the dataframe
dataset = telecom_churn.copy()

In [ ]:
# Display the first five rows of the dataset
dataset.head()

,customerID,SeniorCitizen,Partner,Dependents,tenure,PhoneService,InternetService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST0000,0,No,Yes,23,No,No,Month-to-month,Bank transfer,49.85,1146.55,No
1,CUST0001,0,Yes,No,43,No,DSL,Month-to-month,Mailed check,100.70,4330.10,Yes
2,CUST0002,1,No,No,51,Yes,DSL,One year,Electronic check,97.33,4963.83,Yes
3,CUST0003,1,No,No,72,Yes,DSL,Month-to-month,Credit card,101.38,7299.36,No
4,CUST0004,1,No,No,25,Yes,DSL,Month-to-month,Electronic check,52.22,1305.50,Yes


In [ ]:
# Display the number of rows and columns in the dataset
dataset.shape

(5880, 12)

In [ ]:
# Display the column names of the dataset
dataset.columns

Index(['customerID', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'InternetService', 'Contract', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

# Exploratory Data Analysis

- We'll not consider the `CustomerId` attribute as it doesn't add any value to the analysis and consequent modeling.


In [ ]:
# Define the target variable for the classification task
target = 'Churn'  # Indicates whether the customer has left the service (Yes/No)

# List of numerical features in the dataset
numeric_features = [
    'SeniorCitizen',    # Whether the customer is a senior citizen (binary: 0 or 1)
    'tenure',           # Number of months the customer has been with the service
    'MonthlyCharges',   # Monthly amount charged to the customer
    'TotalCharges'      # Total amount charged to the customer over time
]

# List of categorical features in the dataset
categorical_features = [
    'Partner',          # Whether the customer has a partner (Yes/No)
    'Dependents',       # Whether the customer has dependents (Yes/No)
    'PhoneService',     # Whether the customer has phone service (Yes/No)
    'InternetService',  # Type of internet service (DSL, Fiber optic, No)
    'Contract',         # Type of contract (Month-to-month, One year, Two year)
    'PaymentMethod'     # Customer's preferred payment method
]

In [45]:
# Generate summary statistics for numerical features
dataset[numeric_features].describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,5880.000000,5880.000000,5880.000000,5880.000000
mean,0.500680,36.549150,70.157779,2566.813165
std,0.500042,20.909674,28.804615,1910.017743
min,0.000000,1.000000,20.000000,20.030000
25%,0.000000,18.000000,45.717500,1020.217500
50%,1.000000,37.000000,70.155000,2136.445000
75%,1.000000,55.000000,95.457500,3767.665000
max,1.000000,72.000000,119.990000,8589.600000


- **SeniorCitizen (Binary: 0 or 1)**  
  - The mean (~0.50) suggests an almost equal distribution of senior and non-senior citizens.  
  - The median (50th percentile) is 1, indicating that more senior citizens might be slightly dominant.  

- **Tenure (Months with the Service)**  
  - Customers have a wide range of tenure (1 to 72 months).  
  - The median is 37 months, meaning at least half the customers have stayed for little over three years.  
  - A standard deviation of ~20.91 shows a high variance in customer retention.  

- **MonthlyCharges (Recurring Charges)**  
  - The average monthly charge is around 70.16.  
  - Charges range from 20 to nearly 120, showing diverse pricing plans.  
  - The 25th percentile (45.72) and 75th percentile (95.46) indicate that most customers pay between these amounts.  

- **TotalCharges (Cumulative Payments)**  
  - The average total charge is ~2566.81, but it varies significantly (std ~1910.02).  
  - Some customers have only paid 20.03, while others have spent up to $8589.60, reflecting tenure differences.  

**Key Observations:**  
- Tenure distribution suggests a mix of new and long-term customers.  
- Higher variability in **MonthlyCharges** and **TotalCharges** indicates diverse customer segments.  
- Senior citizens might have slightly longer tenure, impacting retention strategies.

In [ ]:
dataset[categorical_features].describe()

,Partner,Dependents,PhoneService,InternetService,Contract,PaymentMethod
count,5880,5880,5880,5880,5880,5880
unique,2,2,2,3,3,4
top,Yes,No,Yes,No,Month-to-month,Credit card
freq,2995,3004,2941,2029,2020,1503


**Categorical Feature Insights:**  

- **Partner (Yes/No)**  
  - Nearly half of the customers have a partner (**Yes: 2995, No: 2885**).  
  - A slight majority do not have a partner, which may correlate with churn behavior.  

- **Dependents (Yes/No)**  
  - More customers do **not** have dependents (**No: 3004, Yes: 2876**).  
  - This could indicate that independent customers might have different service needs or higher churn rates.  

- **PhoneService (Yes/No)**  
  - Most customers have phone service (**Yes: 2941, No: 2939**), showing an almost even split.  

- **InternetService (DSL, Fiber optic, No)**  
  - The most common category is **No Internet Service (2029 customers)**.  
  - The remaining customers are split between **DSL and Fiber optic**, suggesting that a significant portion of the customer base relies only on phone service.  

- **Contract (Month-to-month, One year, Two year)**  
  - The majority of customers are on **Month-to-month contracts (2020 customers)**, which is often linked to higher churn rates.  
  - Fewer customers commit to longer-term contracts (**One-year and Two-year**).  

- **PaymentMethod (Credit card, Electronic check, Mailed check, Bank transfer)**  
  - **Credit card** is the most used payment method (**1503 customers**), followed by other methods.  
  - Payment method choice could indicate customer preferences and stability—e.g., electronic checks may be linked to higher churn.  

**Key Observations:**  
- A high number of **month-to-month contract customers** suggests potential churn risks.  
- Many customers **do not have internet service**, implying reliance on phone services.  
- **Dependents and partner status** could impact retention strategies, as independent customers might be more likely to switch providers.

In [ ]:
# Compute the proportion of each class in the target variable
dataset[target].value_counts(normalize=True)

,proportion
Churn,
No,0.507143
Yes,0.492857


- **50.7% customers did not churn (No)**  
- **49.3% customers churned (Yes)**  


# Model Training with Hyperparameter Tuning

## Data Preprocessing

In [ ]:
# Define predictor matrix (X) using selected numeric and categorical features
X = dataset[numeric_features + categorical_features]

# Define target variable
y = dataset[target].map({'No': 0, 'Yes': 1})

In [ ]:
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,              # Predictors (X) and target variable (y)
    test_size=0.2,     # 20% of the data is reserved for testing
    random_state=42    # Ensures reproducibility by setting a fixed random seed
)

In [ ]:
# Create a preprocessing pipeline for numerical and categorical features
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),  # Scale numeric features to have mean equal to 0 and standard deviation equal to 1
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)  # Encode categorical features as one-hot vectors
)

- ``make_column_transformer`` is a function from scikit-learn that helps create a preprocessing pipeline for applying different transformations to specific feature types in a dataset

**Breakdown:**

1. **`StandardScaler()` for numeric features**  
   - Scales numerical columns (**`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`**) to have **zero mean and unit variance**.  
   - This ensures that numerical features with different ranges are transformed to the same scale, improving model performance.  

2. **`OneHotEncoder()` for categorical features**  
   - Converts categorical columns (**`Partner`, `Dependents`, `PhoneService`, `InternetService`, `Contract`, `PaymentMethod`**) into **one-hot encoded vectors**.  
   - `handle_unknown='ignore'` ensures that any new categories encountered during inference don’t cause errors.  

This preprocessing step prepares the dataset for machine learning models by standardizing numerical values and encoding categorical data efficiently.

## Creating Model Pipeline

In [ ]:
# Initialize an XGBoost classifier
model_xgb = xgb.XGBClassifier(random_state=42)

In [ ]:
# Create a machine learning pipeline with preprocessing and model training steps
model_pipeline = make_pipeline(
    preprocessor,  # Preprocesses numerical and categorical features
    model_xgb      # XGBoost classifier for model training
)

**Explanation of `make_pipeline`**  

`make_pipeline` is a function from **scikit-learn** that creates a sequential pipeline for **data preprocessing and model training**. It ensures that all necessary transformations are applied before passing the data to the model.  

**What Happens Here?**
1. **`preprocessor`** (created using `make_column_transformer`)  
   - Standardizes numerical features (e.g., `StandardScaler`)  
   - One-hot encodes categorical features (e.g., `OneHotEncoder`)  

2. **`model_xgb` (XGboost classifier)**  
   - Trains a machine learning model (XGBoost here) **after preprocessing**  
   - Ensures that raw data is first transformed before being passed to the model  

## Model Training

In [ ]:
# Train the model pipeline on the training data
model_pipeline.fit(Xtrain, ytrain)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Partner', 'Dependents',
                                                   'PhoneService',
                                                   'InternetService',
                                                   'Contract',
                                                   'PaymentMethod'])])),
                ('xgbclassifier',
                 XGBClassifier(base_score...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

- One can click on each dropdown button above the pipeline architecture to view the backend process that occurs when you run the model_pipeline on the dataset.

In [ ]:
# Fit the pipeline to the training data first
model_pipeline.fit(Xtrain, ytrain)

# Make predictions on the training data
y_pred_train = model_pipeline.predict(Xtrain)

# Make predictions on the test data
y_pred_test = model_pipeline.predict(Xtest)

In [ ]:
# Generate a classification report to evaluate model performance on training set
print(classification_report(ytrain, y_pred_train))

              precision    recall  f1-score   support

           0       0.92      0.93      0.92      2398
           1       0.93      0.91      0.92      2306

    accuracy                           0.92      4704
   macro avg       0.92      0.92      0.92      4704
weighted avg       0.92      0.92      0.92      4704



In [ ]:
# Generate a classification report to evaluate model performance on test set
print(classification_report(ytest, y_pred_test))

              precision    recall  f1-score   support

           0       0.49      0.52      0.50       584
           1       0.50      0.47      0.49       592

    accuracy                           0.49      1176
   macro avg       0.50      0.50      0.49      1176
weighted avg       0.50      0.49      0.49      1176



- The model achieves **49% accuracy**, with similar precision, recall, and F1-scores for both classes.
- Class 0 has **52% recall**, while class 1 has **47% recall**, indicating the model struggles to balance predictions.
- The overall difference in performance between train and test sets points out that the model is overfitting, and suggests the need for hyperparameter tuning.

## Hyperparameter Tuning

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define feature lists to ensure they are loaded in memory
numeric_features = [
    'SeniorCitizen',    # Whether the customer is a senior citizen (binary: 0 or 1)
    'tenure',           # Number of months the customer has been with the service
    'MonthlyCharges',   # Monthly amount charged to the customer
    'TotalCharges'      # Total amount charged to the customer over time
]

categorical_features = [
    'Partner',          # Whether the customer has a partner (Yes/No)
    'Dependents',       # Whether the customer has dependents (Yes/No)
    'PhoneService',     # Whether the customer has phone service (Yes/No)
    'InternetService',  # Type of internet service (DSL, Fiber optic, No)
    'Contract',         # Type of contract (Month-to-month, One year, Two year)
    'PaymentMethod'     # Customer's preferred payment method
]

# Define the preprocessing steps
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

In [ ]:
# Define base XGBoost model
xgb_model = xgb.XGBClassifier(random_state=42)

# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 100, 150, 200],    # number of tree to build
    'xgbclassifier__max_depth': [2, 3, 4],    # maximum depth of each tree
    'xgbclassifier__colsample_bytree': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each tree
    'xgbclassifier__colsample_bylevel': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each level of a tree
    'xgbclassifier__learning_rate': [0.01, 0.05, 0.1],    # learning rate
    'xgbclassifier__reg_lambda': [0.4, 0.5, 0.6],    # L2 regularization factor
}

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

# Try to find the file in common locations
drive_path = "/content/drive/MyDrive/customer_churn_data.csv"
local_path = "/content/customer_churn_data.csv"

if os.path.exists(local_path):
    print("Loading dataset from local path...")
    telecom_churn = pd.read_csv(local_path)
elif os.path.exists(drive_path):
    print("Loading dataset from Google Drive...")
    telecom_churn = pd.read_csv(drive_path)
else:
    raise FileNotFoundError(
        "Could not find 'customer_churn_data.csv'. Please upload the file directly to "
        "the file pane in Colab (so it resides in '/content/customer_churn_data.csv') or ensure "
        "it is in your Google Drive at 'MyDrive/customer_churn_data.csv'."
    )

dataset = telecom_churn.copy()

# Prepare features and target variable
X = dataset[numeric_features + categorical_features]
y = dataset[target].map({'No': 0, 'Yes': 1})

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Create the tuning pipeline combining preprocessing and the model
tuning_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgbclassifier', xgb_model)
])

# Streamlined parameter grid for quick execution (only 4 candidates, 12 fits total)
quick_param_grid = {
    'xgbclassifier__n_estimators': [50, 100],
    'xgbclassifier__max_depth': [2, 3],
    'xgbclassifier__learning_rate': [0.1]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=quick_param_grid,
    scoring='recall',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# Fit the grid search to find the best model parameters
print("Starting fast GridSearchCV model fitting...")
grid_search.fit(Xtrain, ytrain)
print("GridSearchCV model fitting complete!")

Loading dataset from Google Drive...
Starting fast GridSearchCV model fitting...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
GridSearchCV model fitting complete!


**Note**: The following code, which includes model pipeline creation and hyperparameter tuning using GridSearchCV, will take ~10-15 minutes to complete. Please allow sufficient time for execution.

## Selecting the Best Model

In [ ]:
# Check if the grid search completed successfully and display parameters
try:
    print("Best Parameters:", grid_search.best_params_)
except AttributeError:
    print("Error: The GridSearchCV process has not completed successfully yet.")
    print("Please re-run cell 'bcfec192' and allow it to finish training without interruption.")

Best Parameters: {'xgbclassifier__learning_rate': 0.1, 'xgbclassifier__max_depth': 3, 'xgbclassifier__n_estimators': 100}


In [ ]:
# Store the best model
best_model = grid_search.best_estimator_
best_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Partner', 'Dependents',
                                                   'PhoneService',
                                                   'InternetService',
                                                   'Contract',
                                                   'PaymentMethod'])])),
                ('xgbclassifier',
                 XGBClassifier(base_score=None...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

The classification threshold is important because it controls precision and recall trade-offs.
- Since customer churn prediction is a retention problem, a higher recall is preferred. We want to correctly identify as many churners as possible, even if it means getting some false positives.
- We'll lower the classification threshold to 0.45 from 0.5 to increase the recall.

In [ ]:
# Set the classification threshold
classification_threshold = 0.45

In [ ]:
# Make predictions on the training data
y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

# Make predictions on the test data
y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

In [ ]:
# Generate a classification report to evaluate model performance on training set
print(classification_report(ytrain, y_pred_train))

              precision    recall  f1-score   support

           0       0.73      0.32      0.44      2398
           1       0.55      0.88      0.68      2306

    accuracy                           0.59      4704
   macro avg       0.64      0.60      0.56      4704
weighted avg       0.65      0.59      0.56      4704



In [ ]:
# Generate a classification report to evaluate model performance on test set
print(classification_report(ytest, y_pred_test))

              precision    recall  f1-score   support

           0       0.49      0.20      0.29       584
           1       0.50      0.79      0.61       592

    accuracy                           0.50      1176
   macro avg       0.50      0.50      0.45      1176
weighted avg       0.50      0.50      0.45      1176



- We can see that the **overfitting has significantly reduced**.
- The **test set recall for the class corresponding to churn** has also **significantly improved** (by ~25%) to 78%.
- As expected, while recall has improved, precision has dropped.

We'll go ahead with this model as our final model.

# Model Serialization

In [ ]:
import os

# Create a folder for storing the files needed for web app deployment
os.makedirs("deployment_files", exist_ok=True)

In [ ]:
import joblib

# Define the file path to save (serialize) the trained model along with the data preprocessing steps
saved_model_path = "deployment_files/churn_prediction_model_v1_0.joblib"

# Save the trained model pipeline using joblib
joblib.dump(best_model, saved_model_path)

['deployment_files/churn_prediction_model_v1_0.joblib']

This code is used to save a trained machine learning model pipeline using `joblib`, which is a library for efficient object serialization in Python.

**Breakdown:**  
1. **`saved_model_path = "churn_prediction_model_v1_0.joblib"`**  
   - Defines the file path where the model will be saved.  
   - The model will be stored as a `.joblib` file, a format optimized for large NumPy arrays and machine learning models.  
   - The last part of the filename (`v1_0`) specifies a version number, which is a good practice to track changes and maintain multiple model iterations

2. **`joblib.dump(model_pipeline, saved_model_path)`**  
   - Saves the trained `model_pipeline` object to the specified path (`model.joblib`).  
   - `joblib.dump()` is preferred over `pickle.dump()` for saving large models because it is faster and more efficient with numerical data.  
   - The saved file can be loaded later using `joblib.load("model.joblib")` for inference or further training.  

This approach ensures that the model pipeline, including preprocessing steps and the trained model, is preserved for later use.

In [ ]:
# Load the saved model pipeline from the file
saved_model = joblib.load("deployment_files/churn_prediction_model_v1_0.joblib")

1. **`joblib.load("churn_prediction_model_v1_0.joblib")`**  
   - Loads the previously saved machine learning model (or pipeline) from the `model.joblib` file.  
   - The model retains all trained parameters, including preprocessing steps and learned patterns.

2. **`saved_model`**  
   - This variable stores the deserialized model, allowing it to be used for inference, further training, or evaluation.

This allows you to reuse the trained model **without retraining it.**

In [ ]:
saved_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Partner', 'Dependents',
                                                   'PhoneService',
                                                   'InternetService',
                                                   'Contract',
                                                   'PaymentMethod'])])),
                ('xgbclassifier',
                 XGBClassifier(base_score=None...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

Let's try making predictions on the test set using the deserialized model.

- Please ensure that the saved model is loaded before making predictions.

In [ ]:
saved_model.predict(Xtest)

array([1, 0, 1, ..., 1, 0, 1])

- As we can see, the model can be directly used for making predictions without any retraining.

# Creating a Web App using Streamlit

We want to create a web app using Streamlit that can do the following:
1. Create a UI for users to provide their input
2. Load a serialized ML model
3. Take the user input and loaded model to make a prediction
4. Display the prediction from the model to the user

For this, we write an **`app.py`** script using the %%writefile command that'll do all the above steps in one shot.

**What is `%%writefile deployment_files/app.py`?**  

- `%%writefile` is a **magic command**  that **writes the following cell’s contents to a file**.  
- `deployment_files/app.py`: This means the contents of the cell will be saved as a Python file (`app.py`) inside the `deployment_files/` directory.

In [ ]:
import joblib
import pandas as pd
import gradio as gr

# ZeroGPU Spaces refuse to start unless at least one @spaces.GPU function exists.
# This model runs on CPU, so we add a tiny function that is never called.
# (The `spaces` package is preinstalled on Hugging Face; it's skipped locally.)
try:
    import spaces

    @spaces.GPU(duration=1)
    def _gpu_probe():
        return "ok"
except ImportError:
    pass

# Correct path to the trained model inside deployment_files
model = joblib.load("deployment_files/churn_prediction_model_v1_0.joblib")

def predict(senior, partner, dependents, phone, internet, contract,
            payment, tenure, monthly, total):
    # Build one row with the same columns used in training
    row = pd.DataFrame([{
        "SeniorCitizen": 1 if senior == "Yes" else 0,   # adjust if your model expects "Yes Nord"
        "Partner": partner,
        "Dependents": dependents,
        "PhoneService": phone,
        "InternetService": internet,
        "Contract": contract,
        "PaymentMethod": payment,
        "tenure": tenure,
        "MonthlyCharges": monthly,
        "TotalCharges": total,
    }])

    prob = float(model.predict_proba(row)[0][1])
    verdict = ("High risk: this customer is likely to churn."
               if prob >= 0.5 else
               "Low risk: this customer is likely to stay.")
    return {"Churn": prob, "Stay": 1 - prob}, verdict

yes_no = ["Yes", "No"]

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Dropdown(yes_no, value="No", label="Senior citizen"),
        gr.Dropdown(yes_no, value="No", label="Does the customer have a partner?"),
        gr.Dropdown(yes_no, value="No", label="Does the customer have dependents?"),
        gr.Dropdown(yes_no, value="Yes", label="Does the customer have phone service?"),
        gr.Dropdown(["DSL", "Fiber optic", "No"], value="DSL", label="Type of internet service"),
        gr.Dropdown(["Month-to-month", "One year", "Two year"], value="Month-to-month", label="Type of contract"),
        gr.Dropdown(
            ["Electronic check", "Mailed check",
             "Bank transfer (automatic)", "Credit card (automatic)"],
            value="Electronic check", label="Payment method",
        ),
        gr.Number(value=12, minimum=0, label="Tenure (months with the company)"),
        gr.Number(value=50.0, minimum=0, label="Monthly charges"),
        gr.Number(value=600.0, minimum=0, label="Total charges"),
    ],
    outputs=[
        gr.Label(label="Churn probability"),
        gr.Textbox(label="Risk assessment"),
    ],
    title="Telecom Customer Churn Prediction",
    description="Enter the customer's details to estimate their risk of churning.",
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6cba8748151247ca5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In Streamlit, the methods used to take user inputs in your code are:  

### 1. **`st.number_input()`** – Numeric Input  
   - Allows users to enter numerical values (integers or floats).  
   - You can specify **minimum**, **maximum**, and **default values**.  
   - Example:  
     ```python
     CreditScore = st.number_input("Credit Score", min_value=300, max_value=900, value=650)
     ```
   - Here, the input must be between **300 and 900**, and the default is **650**.  

### 2. **`st.selectbox()`** – Dropdown Selection  
   - Displays a **dropdown menu** for users to choose from predefined options.  
   - Returns the selected option as a string.  
   - Example:  
     ```python
     Geography = st.selectbox("Geography", ["France", "Germany", "Spain"])
     ```
   - Users can only select **one** of the listed values.  

### 3. **`st.button()`** – Button for Action  
   - Displays a button that, when clicked, triggers a specific action.  
   - Example:  
     ```python
     if st.button("Predict"):
         st.write("Prediction process starts...")
     ```
   - When the user clicks **"Predict"**, the code inside the `if` block executes.  

\

**Note**: Please refer to this [link](https://docs.streamlit.io/develop/api-reference/widgets) to check out and experiment with more Streamlit widgets.

# Creating a Dependencies File

In [ ]:
%%writefile deployment_files/requirements.txt
import xgboost, sklearn, pandas, joblib
print(xgboost.__version__, sklearn.__version__, pandas.__version__, joblib.__version__)

Overwriting deployment_files/requirements.txt


A **`requirements.txt`** file is essential for ensuring that your project runs smoothly across different environments. It's like a **blueprint** for setting up your ML project!

# Uploading Files to Hugging Face Repository

Once create the following files in the notebook, lets upload it in to the hugging face space
- **`churn_prediction_model_v1_0.joblib`**
- **`requirements.txt`**
- **`app.py`**

In [ ]:
access_key = "hf_nZxxx"  # Your Hugging Face token created from access keys in write mode
repo_id = "antmourad/churn-app"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/deployment_files",  # Local folder path in azureml
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

CommitInfo(commit_url='https://huggingface.co/spaces/antmourad/churn-app/commit/cc94579008b5638d86a6d17eab5a237ec973d7d7', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cc94579008b5638d86a6d17eab5a237ec973d7d7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/antmourad/churn-app', endpoint='https://huggingface.co', repo_type='space', repo_id='antmourad/churn-app'), pr_revision=None, pr_num=None)

1. **Define authentication and repository details:**  
   - `hf_token` stores the Hugging Face API token for authentication.  
   - `repo_id` specifies the Hugging Face **Space** repository where files will be uploaded.  

2. **Authenticate with Hugging Face:**  
   - The `login(token=hf_token)` function logs into Hugging Face using the provided API token.  

3. **Initialize Hugging Face API object:**  
   - `api = HfApi()` creates an instance of `HfApi`, which allows interaction with the Hugging Face Hub.  

4. **Upload files from the local folder to Hugging Face Space:**  
   - `api.upload_folder()` uploads all files from the `deployment_files` folder to the specified Hugging Face repository.  
   - `folder_path="/content/deployment_files"` specifies the local directory containing the files.  
   - `repo_id=repo_id` sets the target Hugging Face **Space** repository.  
   - `repo_type="space"` ensures that the upload is directed to a **Space** repository, which is used for hosting applications like Streamlit.  


<font size=4>The web application should now be available on your Hugging Face Space. Feel free to experiment with different values and observe the way they impact the output.</font>

<font size=6 color="blue">Power Ahead!</font>
___